# Exploratory Data Analysis: WTA Matches 2023

This notebook explores a single season of WTA match data — your starting point for understanding
the dataset before we build anything on top of it.

Data: [JeffSackmann/tennis_wta](https://github.com/JeffSackmann/tennis_wta) · CC BY-NC-SA 4.0

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from errors import check_multiple_choice, get_notebook_logger, run_guarded

logger = get_notebook_logger()

sns.set_theme(style="whitegrid")  # This sets the theme for seaborn plots to have a white grid background.

## [Notebook Helpers]

This notebook imports helper functions from `errors.py`:
- `get_notebook_logger(...)` sets up consistent logging output.
- `check_multiple_choice(...)` validates MCQ responses with clear feedback.
- `run_guarded(...)` wraps risky steps and prints a learner-friendly error message before re-raising.

You can complete exercises without editing `errors.py`, but if an import fails, check that `errors.py` exists in the project root.

## [ASIDE] DataFrames

A DataFrame is pandas' equivalent of a spreadsheet tab: rows are records, columns are variables.
Two differences worth knowing: it lives in memory rather than on disk, and you interact with it
through code rather than a GUI.

`df.shape` returns `(n_rows, n_columns)` — a useful sanity check every time you load new data.

## Exercise 1: Loading Data

**Which function loads a CSV file into a DataFrame?**

- A) `pd.open_csv()`
- B) `pd.read_csv()`
- C) `pd.load_csv()`
- D) `pd.import_csv()`

Set `answer` to the correct letter in the cell below.

In [ ]:
answer = "B"

options = {
    "A": "open_csv",
    "B": "read_csv",
    "C": "load_csv",
    "D": "import_csv",
}

check_multiple_choice(
    answer=answer,
    options=options,
    is_correct=lambda selected, choices: hasattr(pd, choices[selected]),
    exercise_label="EDA Exercise 1",
    incorrect_message="Not quite - check the pandas docs for reading flat files.",
    logger=logger,
)

## Exercise 2: Load the Data

In [ ]:
FILEPATH = "data/wta_matches_2023.csv"

# TODO: assign load_fn to the correct pandas CSV loader function (function object only).
# Don't call the function or argument here, just assign the function object to load_fn.
load_fn = ...

df = run_guarded(
    step_label="EDA Exercise 2",
    action=lambda: load_fn(FILEPATH),
    user_error_message="Could not load the dataset. Check that load_fn is callable and FILEPATH is correct.",
    logger=logger,
)

df.head()

### If This Fails, Check

- `load_fn` is a function object, not the result of calling the function.
- `FILEPATH` points to an existing file.
- You ran the imports cell first (so `run_guarded` and `logger` exist).
- Your kernel is using the project environment with pandas installed.

## Exercise 3: Inspect the DataFrame

Use `df.shape`, `df.columns`, and `df.dtypes` to get a first look.

In [ ]:
# Complete the three assignments using DataFrame inspection tools.
shape = ...
columns = ...
dtypes = ...

def _exercise_3_output():
    print(f"Shape: {shape}")
    print(f"\nColumns:\n{columns}")
    print(f"\nData types:\n{dtypes}")

run_guarded(
    step_label="EDA Exercise 3",
    action=_exercise_3_output,
    user_error_message="Could not inspect DataFrame metadata. Check your shape/columns/dtypes assignments.",
    logger=logger,
)

## [ASIDE] Why dtypes matter

Python distinguishes numbers from strings the same way SQL distinguishes `INT` from `VARCHAR`.
If a column that should be numeric is stored as `object` (i.e. string), arithmetic fails or
produces confusing results. A dtype check when loading unfamiliar data catches this early.

## Exercise 4: Missing Values

Compute the number of missing values per column, sorted descending. Assign to `missing`, then filter to columns where the count is greater than zero.

In [ ]:
# Build the missing-values summary and assign it to `missing`.
missing = ...

missing_nonzero = run_guarded(
    step_label="EDA Exercise 4",
    action=lambda: missing[missing > 0],
    user_error_message="Could not build the missing-values summary. Check your `missing` expression.",
    logger=logger,
)
missing_nonzero

## [ASIDE] Missing data in this dataset

In this file, missing match statistics (aces, serve percentages, double faults) are mostly from
BJK Cup team-format matches, which don't record individual stats. When we join multiple seasons
in the next notebook, pre-1991 matches add more missing rows. Filtering to complete-stat rows
before modelling is a habit worth building now.

## Exercise 5: Summary Statistics

Compute summary statistics for all numeric columns.

In [ ]:
# Fill with a DataFrame summary method name (string), then call it.
summary_method = "describe"
getattr(df, summary_method)()

## Visualisation

The cell below is a complete, working bar chart — read through it before moving on.
We'll use this same fig/ax pattern throughout.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
surface_counts = df["surface"].value_counts()
sns.barplot(x=surface_counts.index, y=surface_counts.values, ax=ax,
            hue=surface_counts.index, palette="muted", legend=False)
ax.set_title("Matches by Surface (2023)")
ax.set_xlabel("Surface")
ax.set_ylabel("Match Count")
plt.tight_layout()
plt.show()

## Exercise 6: Customise the Chart

The same chart is below with three variables you control. Change them and re-run.

In [ ]:
COLOUR = "coral"    # try: "coral", "seagreen", "mediumpurple"
FIG_SIZE = (8, 5)       # try: (10, 4) or (6, 6)
TITLE = "Matches by Surface (2023)"

fig, ax = plt.subplots(figsize=FIG_SIZE)
surface_counts = df["surface"].value_counts()
sns.barplot(x=surface_counts.index, y=surface_counts.values, ax=ax, color=COLOUR)
ax.set_title(TITLE)
ax.set_xlabel("Surface")
ax.set_ylabel("Match Count")
plt.tight_layout()
plt.show()

## [ASIDE] Choosing a chart type

- **Bar** — comparing category totals (matches per surface)
- **Histogram** — distribution of a numeric variable (match duration)
- **Scatter** — relationship between two numeric variables (winner rank vs loser rank)

Same logic as any BI tool. The matplotlib/seaborn layer just makes it scriptable and reproducible.

## Exercise 7: Match Duration

Plot a histogram of match duration using the `minutes` column.

Steps:
1. Create a figure and axes with `plt.subplots(figsize=(8, 5))`
2. Plot a histogram of `df["minutes"]` — remember to drop NaNs first, use 30 bins
3. Set a title and axis labels
4. Call `plt.tight_layout()` and `plt.show()`

In [ ]:
# Step 1: create figure and axes (figsize=(8, 5))
fig, ax = plt.subplots(figsize=(8, 5))

# Step 2: plot a histogram of df["minutes"] — drop NaNs first
# hint: .dropna().plot(kind="hist", bins=30, ax=ax, ...)
df["minutes"].dropna().plot(kind="hist", bins=30, ax=ax, color="coral")

# Step 3: add a title and axis labels
ax.set_title("Match Duration (Minutes)")
ax.set_xlabel("Minutes")
ax.set_ylabel("Frequency")

# Step 4: plt.tight_layout() and plt.show()
plt.tight_layout()
plt.show()

## Exercise 8: Winner vs Loser Rank

Plot a scatter: winner rank on x, loser rank on y, coloured by surface.

Steps:
1. Create figure and axes with `plt.subplots(figsize=(8, 6))`
2. Loop over `df.groupby("surface")` — each iteration gives `(surface_name, group_df)`
3. In the loop, call `ax.scatter(group["winner_rank"], group["loser_rank"], label=surface, alpha=0.4, s=20)`
4. After the loop: add `ax.legend(title="Surface")`, a title, and axis labels

In [ ]:
# Step 1: create figure and axes (figsize=(8, 6))
fig, ax = plt.subplots(figsize=(8, 6))

# Step 2: write: for surface, group in df.groupby("surface"):
for surface, group in df.groupby("surface"):

    # Step 3: inside the loop — scatter plot: winner_rank on x, loser_rank on y
    # use alpha=0.4, s=20, and pass label=surface
    ax.scatter(group["winner_rank"], group["loser_rank"], alpha=0.4, s=20, label=surface)

# Step 4: after the loop — add legend (title="Surface"), title, and axis labels
ax.legend(title="Surface")
ax.set_title("Winner vs Loser Rank by Surface")
ax.set_xlabel("Winner Rank")
ax.set_ylabel("Loser Rank")
plt.tight_layout()
plt.show()

## Exercise 9: Most Match Wins

Who won the most matches in 2023? Use `value_counts()` on the `winner_name` column and take
the top 10 with `.head(10)`.

In [ ]:
# Use value_counts() on the winner_name column, take top 10
top_winners = df["winner_name"].value_counts().head(10)
top_winners

## Exercise 10: Write a Function

Complete the function below. It should return the top `n` players by wins as a DataFrame
with columns `["player", "wins"]`, sorted descending.

Fill in the docstring and the function body.

In [ ]:
def top_n_players(df: pd.DataFrame, n: int) -> pd.DataFrame:
    """
    Returns the top n players by number of wins.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing match data with a "winner_name" column.
    n : int
        Number of top players to return.

    Returns
    -------
    pd.DataFrame
        DataFrame containing the top n players and their win counts.
    """
    top_players = df["winner_name"].value_counts().head(n)
    return top_players


# Once implemented, this should display the top 10 players by wins
top_n_players(df, n=10)

## Git Signpost

Good place to commit. In VS Code open Source Control (`Ctrl+Shift+G` / `Cmd+Shift+G`),
stage `EDA.ipynb`, and write a short commit message.

Or in the terminal:
```bash
git add EDA.ipynb
git commit -m "complete EDA notebook"
```